In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
import pandas as pd
from thop import profile

# ─────────────────────────────────────────
# 0. SEED
# ─────────────────────────────────────────
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    np.random.seed(42 + worker_id)
    random.seed(42 + worker_id)

set_seed(42)

# ─────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────
INPUT_PATH = "input path"
OUTPUT_DIR  = "/kaggle/working/ablation_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

EPOCHS      = 30 
BATCH_SIZE  = 32
LR          = 1e-4
IMG_SIZE    = (224, 224)
NUM_CLASSES = 3
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# ─────────────────────────────────────────
# 1. DATASET  —  stratified split
# ─────────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

full_dataset = datasets.ImageFolder(INPUT_PATH, transform=transform)
class_names  = full_dataset.classes
targets      = np.array(full_dataset.targets)
all_indices  = np.arange(len(full_dataset))

# stratified 70 / 15 / 15
train_idx, temp_idx = train_test_split(
    all_indices, test_size=0.30, stratify=targets, random_state=42
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, stratify=targets[temp_idx], random_state=42
)

train_set = Subset(full_dataset, train_idx)
val_set   = Subset(full_dataset, val_idx)
test_set  = Subset(full_dataset, test_idx)

print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")

g = torch.Generator(); g.manual_seed(42)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, worker_init_fn=seed_worker, generator=g)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# class weights — guaranteed all classes present after stratified split
train_labels    = targets[train_idx]
class_weights   = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=train_labels)
class_weights_t = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
print("Class weights:", class_weights)

# ─────────────────────────────────────────
# 2. BUILDING BLOCKS
# ─────────────────────────────────────────
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.size()
        y = x.mean(dim=(2,3))
        y = self.fc(y).view(b, c, 1, 1)
        return x * y


class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels)
        )
        self.conv    = nn.Conv2d(2, 1, kernel_size=kernel_size,
                                 padding=kernel_size // 2)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_p = x.mean(dim=(2,3))
        max_p = torch.amax(x, dim=(2,3))
        ca    = torch.sigmoid(self.mlp(avg_p) + self.mlp(max_p))
        x     = x * ca.view(x.size(0), x.size(1), 1, 1)
        sa    = torch.cat([x.mean(dim=1,keepdim=True),
                           torch.amax(x,dim=1,keepdim=True)], dim=1)
        sa    = self.sigmoid(self.conv(sa))
        return x * sa


def unfreeze_last_n(model, n):
    for p in model.parameters():
        p.requires_grad = False
    for p in list(model.parameters())[-n:]:
        p.requires_grad = True

# ─────────────────────────────────────────
# 3. ABLATION VARIANTS
# ─────────────────────────────────────────

class DenseNetOnly(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = models.densenet121(pretrained=True)
        self.backbone.classifier = nn.Identity()
        unfreeze_last_n(self.backbone, 50)
        self.classifier = nn.Sequential(
            nn.Linear(1024, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.backbone(x))


class EfficientNetOnly(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = models.efficientnet_b0(pretrained=True)
        self.backbone.classifier = nn.Identity()
        self.classifier = nn.Sequential(
            nn.Linear(1280, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.backbone(x))


class FusionOnly(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.densenet = models.densenet121(pretrained=True)
        self.densenet.classifier = nn.Identity()
        unfreeze_last_n(self.densenet, 50)

        self.efficientnet = models.efficientnet_b0(pretrained=True)
        self.efficientnet.classifier = nn.Identity()
        for p in self.efficientnet.parameters():
            p.requires_grad = False

        self.head_d = nn.Sequential(
            nn.Linear(1024,512), nn.BatchNorm1d(512), nn.Dropout(0.3), nn.ReLU(),
            nn.Linear(512,256),  nn.BatchNorm1d(256), nn.Dropout(0.3), nn.ReLU()
        )
        self.head_e = nn.Sequential(
            nn.Linear(1280,512), nn.BatchNorm1d(512), nn.Dropout(0.3), nn.ReLU(),
            nn.Linear(512,256),  nn.BatchNorm1d(256), nn.Dropout(0.3), nn.ReLU()
        )
        self.classifier = nn.Sequential(
            nn.Linear(512,128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        fd = self.head_d(self.densenet(x))
        fe = self.head_e(self.efficientnet(x))
        return self.classifier(torch.cat([fd, fe], dim=1))


class FusionSEOnly(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.densenet = models.densenet121(pretrained=True)
        self.densenet.classifier = nn.Identity()
        unfreeze_last_n(self.densenet, 50)
        self.se_d = SEBlock(1024)

        self.efficientnet = models.efficientnet_b0(pretrained=True)
        self.efficientnet.classifier = nn.Identity()
        for p in self.efficientnet.parameters():
            p.requires_grad = False
        self.se_e = SEBlock(1280)

        self.head_d = nn.Sequential(
            nn.Linear(1024,512), nn.BatchNorm1d(512), nn.Dropout(0.3), nn.ReLU(),
            nn.Linear(512,256),  nn.BatchNorm1d(256), nn.Dropout(0.3), nn.ReLU()
        )
        self.head_e = nn.Sequential(
            nn.Linear(1280,512), nn.BatchNorm1d(512), nn.Dropout(0.3), nn.ReLU(),
            nn.Linear(512,256),  nn.BatchNorm1d(256), nn.Dropout(0.3), nn.ReLU()
        )
        self.classifier = nn.Sequential(
            nn.Linear(512,128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        fd = self.densenet(x).unsqueeze(-1).unsqueeze(-1)
        fd = self.se_d(fd).squeeze(-1).squeeze(-1)
        fd = self.head_d(fd)

        fe = self.efficientnet(x).unsqueeze(-1).unsqueeze(-1)
        fe = self.se_e(fe).squeeze(-1).squeeze(-1)
        fe = self.head_e(fe)

        return self.classifier(torch.cat([fd, fe], dim=1))


class CXRNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.densenet = models.densenet121(pretrained=True)
        self.densenet.classifier = nn.Identity()
        unfreeze_last_n(self.densenet, 50)
        self.se_d = SEBlock(1024)

        self.efficientnet = models.efficientnet_b0(pretrained=True)
        self.efficientnet.classifier = nn.Identity()
        for p in self.efficientnet.parameters():
            p.requires_grad = False
        self.se_e = SEBlock(1280)

        self.head_d = nn.Sequential(
            nn.Linear(1024,512), nn.BatchNorm1d(512), nn.Dropout(0.3), nn.ReLU(),
            nn.Linear(512,256),  nn.BatchNorm1d(256), nn.Dropout(0.3), nn.ReLU()
        )
        self.head_e = nn.Sequential(
            nn.Linear(1280,512), nn.BatchNorm1d(512), nn.Dropout(0.3), nn.ReLU(),
            nn.Linear(512,256),  nn.BatchNorm1d(256), nn.Dropout(0.3), nn.ReLU()
        )
        self.cbam = CBAM(512)
        self.classifier = nn.Sequential(
            nn.Linear(512,128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        fd = self.densenet(x).unsqueeze(-1).unsqueeze(-1)
        fd = self.se_d(fd).squeeze(-1).squeeze(-1)
        fd = self.head_d(fd)

        fe = self.efficientnet(x).unsqueeze(-1).unsqueeze(-1)
        fe = self.se_e(fe).squeeze(-1).squeeze(-1)
        fe = self.head_e(fe)

        f  = torch.cat([fd, fe], dim=1).unsqueeze(-1).unsqueeze(-1)
        f  = self.cbam(f).squeeze(-1).squeeze(-1)
        return self.classifier(f)


# ─────────────────────────────────────────
# 4. TRAIN / EVAL HELPERS
# ─────────────────────────────────────────
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct = 0.0, 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out  = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X.size(0)
        correct    += (out.argmax(1) == y).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0.0, 0
    all_true, all_pred  = [], []
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            out  = model(X)
            loss = criterion(out, y)
            preds = out.argmax(1)
            total_loss += loss.item() * X.size(0)
            correct    += (preds == y).sum().item()
            all_true.extend(y.cpu().numpy())
            all_pred.extend(preds.cpu().numpy())
    return (total_loss / len(loader.dataset),
            correct / len(loader.dataset),
            all_true, all_pred)


def count_params(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def get_flops(model):
    try:
        dummy = torch.randn(1, 3, 224, 224).to(DEVICE)
        flops, _ = profile(model, inputs=(dummy,), verbose=False)
        return flops / 1e9
    except Exception:
        return None


# ─────────────────────────────────────────
# 5. RUN ALL ABLATION VARIANTS
# ─────────────────────────────────────────
variants = {
    "DenseNet121 Only"        : DenseNetOnly(NUM_CLASSES),
    "EfficientNet-B0 Only"    : EfficientNetOnly(NUM_CLASSES),
    "Fusion (No SE, No CBAM)" : FusionOnly(NUM_CLASSES),
    "Fusion + SE (No CBAM)"   : FusionSEOnly(NUM_CLASSES),
    "CXRNet (Full)"           : CXRNet(NUM_CLASSES),
}

results_summary = []

for name, model in variants.items():
    print(f"\n{'='*55}")
    print(f"  VARIANT: {name}")
    print(f"{'='*55}")

    set_seed(42)
    model = model.to(DEVICE)

    criterion = nn.CrossEntropyLoss(weight=class_weights_t)
    optimizer = optim.Adam(
        [p for p in model.parameters() if p.requires_grad], lr=LR
    )

    total_p, trainable_p = count_params(model)
    gflops = get_flops(model)

    best_val_acc  = 0.0
    best_state    = None
    train_accs, val_accs = [], []

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        vl_loss, vl_acc, _, _ = evaluate(model, val_loader, criterion)
        train_accs.append(tr_acc)
        val_accs.append(vl_acc)
        print(f"  Ep {epoch}/{EPOCHS} | "
              f"Train Acc: {tr_acc:.4f} Loss: {tr_loss:.4f} | "
              f"Val Acc: {vl_acc:.4f} Loss: {vl_loss:.4f}")

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_state   = {k: v.clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    _, test_acc, y_true, y_pred = evaluate(model, test_loader, criterion)

    report = classification_report(
        y_true, y_pred, target_names=class_names, digits=4, output_dict=True
    )
    macro_p  = report["macro avg"]["precision"]
    macro_r  = report["macro avg"]["recall"]
    macro_f1 = report["macro avg"]["f1-score"]

    print(f"\n  ✅ Test Acc: {test_acc:.4f} | "
          f"Macro P: {macro_p:.4f} | R: {macro_r:.4f} | F1: {macro_f1:.4f}")
    print(f"  Params (Total/Trainable): {total_p:,} / {trainable_p:,}")
    if gflops:
        print(f"  GFLOPs: {gflops:.2f}")

    results_summary.append({
        "Variant"             : name,
        "Test Accuracy (%)"   : round(test_acc * 100, 2),
        "Macro Precision (%)" : round(macro_p  * 100, 2),
        "Macro Recall (%)"    : round(macro_r  * 100, 2),
        "Macro F1 (%)"        : round(macro_f1 * 100, 2),
        "Total Params (M)"    : round(total_p / 1e6, 2),
        "Trainable Params (M)": round(trainable_p / 1e6, 2),
        "GFLOPs"              : round(gflops, 2) if gflops else "N/A",
    })

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d",
                xticklabels=class_names, yticklabels=class_names, cmap="Blues")
    plt.title(f"Confusion Matrix — {name}")
    plt.xlabel("Predicted"); plt.ylabel("True")
    safe_name = name.replace(" ","_").replace("(","").replace(")","").replace(",","")
    plt.savefig(os.path.join(OUTPUT_DIR, f"cm_{safe_name}.png"), dpi=300, bbox_inches="tight")
    plt.close()

    # Accuracy curve
    plt.figure(figsize=(6,4))
    plt.plot(train_accs, label="Train Acc")
    plt.plot(val_accs,   label="Val Acc")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy")
    plt.title(f"Accuracy Curve — {name}")
    plt.legend()
    plt.savefig(os.path.join(OUTPUT_DIR, f"acc_{safe_name}.png"), dpi=300, bbox_inches="tight")
    plt.close()

    del model
    torch.cuda.empty_cache()

# ─────────────────────────────────────────
# 6. SUMMARY TABLE + PLOTS
# ─────────────────────────────────────────
df = pd.DataFrame(results_summary)
df.to_csv(os.path.join(OUTPUT_DIR, "ablation_summary.csv"), index=False)
print("\n\n========== ABLATION STUDY SUMMARY ==========")
print(df.to_string(index=False))

# Macro F1 bar chart
plt.figure(figsize=(10, 5))
bars = plt.bar(df["Variant"], df["Macro F1 (%)"], color="steelblue", edgecolor="black")
plt.ylim(min(df["Macro F1 (%)"]) - 5, 100)
plt.ylabel("Macro F1 (%)")
plt.title("Ablation Study — Macro F1 Score per Variant")
plt.xticks(rotation=20, ha="right")
for bar, val in zip(bars, df["Macro F1 (%)"]):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.2,
             f"{val:.2f}%", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "ablation_macro_f1.png"), dpi=300, bbox_inches="tight")
plt.close()

# Grouped bar — all 4 metrics
metrics = ["Test Accuracy (%)", "Macro Precision (%)", "Macro Recall (%)", "Macro F1 (%)"]
x    = np.arange(len(df))
w    = 0.18
fig, ax = plt.subplots(figsize=(13,6))
for i, m in enumerate(metrics):
    ax.bar(x + i*w, df[m], width=w, label=m)
ax.set_xticks(x + w*1.5)
ax.set_xticklabels(df["Variant"], rotation=20, ha="right")
ax.set_ylabel("Score (%)")
ax.set_title("Ablation Study — All Metrics")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "ablation_all_metrics.png"), dpi=300, bbox_inches="tight")
plt.close()

print(f"\n✅ All results saved to: {OUTPUT_DIR}")